In [17]:
#!pip install -q requests torch bitsandbytes transformers sentencepiece accelerate

In [22]:
!pip install -q --upgrade bitsandbytes requests torch transformers sentencepiece accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 101.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.

In [23]:
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch

# Sign In to Hugging face

In [24]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [25]:
LLAMA = 'meta-llama/Llama-3.1-8B-Instruct'
PHI3 = 'microsoft/Phi-3-mini-4k-instruct'
GEMMA2 = 'google/gemma-2-2b-it'
QWEN2 = 'Qwen/Qwen2.5-7B-Instruct'
MIXTRAL = 'mistralai/Mixtral-8x7B-Instruct-v0.1'

In [26]:
messages = [
    { "role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Tell a joke about software and AI engineers?" }
]

In [27]:
#quantization configuration - allows to load model into memory with less memory consumption
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [28]:
# tokenizer
tokenizer = AutoTokenizer.from_pretrained(LLAMA, trust_remote_code=True)
tokenizer.pad_tokens = tokenizer.eos_token #end_of_sentence
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt")


In [29]:
model = AutoModelForCausalLM.from_pretrained(LLAMA, quantization_config=quant_config, trust_remote_code=True)

ImportError: Using `bitsandbytes` 4-bit quantization requires the latest version of bitsandbytes: `pip install -U bitsandbytes`

In [1]:
memory = model.get_memory_footprint() 1/e6
print(f"Memory footprint : {memory:,.1f} MB")

SyntaxError: invalid syntax (ipython-input-2780695737.py, line 1)

In [ ]:
model

In [ ]:
outputs = model.generate(inputs, max_new_tokens=80)
print(tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
#cleanup
del input output model
torch.cuda.empty_cache()

In [ ]:
#wrapping everything in a function
def generate(model, messages):
  tokenizer = AutoTokenizer.from_pretrained(LLAMA, trust_remote_code=True)
  tokenizer.pad_tokens = tokenizer.eos_token #end_of_sentence
  inputs = tokenizer.apply_chat_template(messages, return_tensors="pt")
  streamer = TextStreamer(tokenizer)
  model = AutoModelForCausalLM.from_pretrained(LLAMA, quantization_config=quant_config, trust_remote_code=True)
  outputs = model.generate(inputs, streamer=streamer, max_new_tokens=80)
  del tokenizer, streamer, model, inputs, outputs
  torch.cuda.empty_cache()

In [ ]:
generate(PHI3, messages)

In [ ]:
messages = [
    {"role": "user", "content": "Tell a joke about software and AI engineers?"}
]
generate(GEMMA2, messages)